In [ ]:
import numpy as np
import networkx as nx
from scipy import sparse
from scipy.sparse.linalg import eigsh
import warnings

from src.analysis.further_measures import (ollivier_ricci_curvature, 
                                           rich_club_coefficient, 
                                           participation_coefficient, 
                                           persistent_homology, 
                                           targeted_attack_robustness, 
                                           algebraic_connectivity) 

In [5]:
from src.ESNs.memory_capacity_weighted import evaluate_memory_capacity_from_connectome

# def evaluate_memory_capacity_from_connectome(connectome: np.ndarray, 
#                                              h_params: Optional[Dict[str, Any]] = None,
                                             
#                                              calculate_criticality:  Optional[bool] = False,
#                                              calculate_info_dynamics: Optional[bool] = False,              
#  ) -> Dict[str, float]:

h_params = {
    "n_runs": 50,
    "train_len": 5000,
    "test_len": 1000,
    "n_lags": 50,
    "spectral_radius": 0.9,
    "input_scaling": 1.0,
    "leak_rate": 1.0,
    "bias": 0.0,
    "regression_method": "pinv",
    "random_state": 42,
    "n_transient": 0
}

res = evaluate_memory_capacity_from_connectome(np.array(A_test, dtype=np.float32), h_params)

/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/numpy/linalg/linalg.py:2030: RuntimeWarning: divide by zero encountered in matmul
  res = matmul(transpose(vt), multiply(s[..., newaxis], transpose(u)))
/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/numpy/linalg/linalg.py:2030: RuntimeWarning: overflow encountered in matmul
  res = matmul(transpose(vt), multiply(s[..., newaxis], transpose(u)))
/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/numpy/linalg/linalg.py:2030: RuntimeWarning: invalid value encountered in matmul
  res = matmul(transpose(vt), multiply(s[..., newaxis], transpose(u)))
/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/echoes/esn/_base.py:312: RuntimeWarning: divide by zero encountered in matmul
  W_out = (np.linalg.pinv(full_states) @ outputs).T
/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/echoes/esn/_base.py:312: RuntimeWarning: overflow encountered in matmul
  W_out = (np.linalg.pinv(full_states) @ outpu

In [ ]:
res

In [2]:

# =============================================================================
# Convenience: compute all at once
# =============================================================================
def compute_additional_properties(A, properties=None):
    """
    Compute all (or selected) additional graph properties.

    Parameters
    ----------
    A : np.ndarray
        Adjacency matrix (N x N).
    properties : list of str, optional
        Subset of properties to compute. If None, computes all.
        Valid values:
            'ollivier_ricci_curvature'
            'rich_club_coefficient'
            'participation_coefficient'
            'persistent_homology'
            'targeted_attack_robustness'
            'algebraic_connectivity'

    Returns
    -------
    dict
        Combined dictionary of all computed property summaries.
    """
    all_funcs = {
        'ollivier_ricci_curvature':   ollivier_ricci_curvature,
        'rich_club_coefficient':      rich_club_coefficient,
        'participation_coefficient':  participation_coefficient,
        'persistent_homology':        persistent_homology,
        'targeted_attack_robustness': targeted_attack_robustness,
        'algebraic_connectivity':     algebraic_connectivity,
    }

    if properties is None:
        properties = list(all_funcs.keys())

    results = {}
    for prop in properties:
        if prop not in all_funcs:
            warnings.warn(f"Unknown property: {prop}. Skipping.")
            continue
        try:
            result = all_funcs[prop](A)
            results.update(result)
        except Exception as e:
            warnings.warn(f"Error computing {prop}: {e}")
            # Still add NaN entries so downstream code doesn't break
            # (we don't know exact keys, so skip)

    return results


# =============================================================================
# Quick test / demo
# =============================================================================
if __name__ == "__main__":
    # Generate a small test graph
    rng = np.random.RandomState(0)
    N = 50
    A_test = rng.rand(N, N)
    A_test = (A_test + A_test.T) / 2
    A_test[A_test < 0.7] = 0  # sparsify
    np.fill_diagonal(A_test, 0)

    print(f"Test graph: {N} nodes, {int(np.sum(A_test > 0) / 2)} edges\n")

    # Compute all
    results = compute_additional_properties(A_test)
    for key, val in sorted(results.items()):
        print(f"  {key:35s} = {val}")

Test graph: 50 nodes, 221 edges



/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_42870/2400946984.py:48: UserWarning: Error computing ollivier_ricci_curvature: GraphRicciCurvature is required. Install via: pip install GraphRicciCurvature
  warnings.warn(f"Error computing {prop}: {e}")


  fiedler_bipartition_balance         = 1.0
  fiedler_value                       = 2.582405929382448
  fiedler_value_norm                  = 0.19858907184328248
  laplacian_spectral_gap              = 0.9303560651609692
  n_communities                       = 5
  pc_frac_connector                   = 0.72
  pc_mean                             = 0.639430886925092
  pc_median                           = 0.6750011358581312
  pc_std                              = 0.1505601489826742
  rc_k_at_max                         = 9
  rc_max_norm                         = 1.1124630627498697
  rc_mean_norm                        = 0.9045852225472488
  rc_regime_frac                      = 0.5384615384615384
  rc_weighted_auc                     = 11.259607893114234
  rob_random_auc                      = 0.4854399999999999
  rob_random_half                     = 0.49800000000000005
  rob_ratio                           = 0.9245220830586687
  rob_targeted_auc                    = 0.44880000000000003


/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_42870/2400946984.py:48: UserWarning: Error computing persistent_homology: Either giotto-tda or ripser is required. Install via: pip install giotto-tda   OR   pip install ripser
  warnings.warn(f"Error computing {prop}: {e}")


In [7]:
"""
Comprehensive computational capacity assessment of network architectures
via Echo State Network (ESN) reservoir computing.

Measures multiple facets of computation:
  - Linear memory capacity (faithful recall of past inputs)
  - Nonlinear capacity (quadratic/cubic transformations of past inputs)
  - Cross-term capacity (interaction between inputs at different lags)
  - Input separation (discrimination between distinct input streams)
  - State richness (dimensionality and entropy of reservoir dynamics)
  - Edge-of-chaos proximity (Lyapunov exponent estimate)

Compatible with any (N, N) adjacency matrix. Designed for batch evaluation
across tens of thousands of networks.

References:
  - Jaeger (2001) - Echo state networks
  - Büsing et al. (2010) - Connectivity, dynamics, and memory in RC
  - Dambre et al. (2012) - Information processing capacity of dynamical systems
  - Suárez et al. (2024) - Connectome-based reservoir computing (conn2res)
"""

import numpy as np
from scipy import linalg
from typing import Optional


def _prepare_reservoir_weights(
    A: np.ndarray,
    spectral_radius: float = 0.95,
    exc_inh_ratio: float = 0.8,
    rng: np.random.RandomState = None,
) -> np.ndarray:
    """
    Normalize adjacency matrix for ESN use.

    For binary matrices: assigns random excitatory/inhibitory signs (Dale's law)
    to create heterogeneous dynamics. For weighted matrices: preserves relative
    weight structure.

    Always rescales to target spectral_radius.
    """
    if rng is None:
        rng = np.random.RandomState(0)

    W = A.copy().astype(np.float64)
    np.fill_diagonal(W, 0)

    if np.allclose(W, 0):
        return W

    # Check if binary (only 0s and 1s)
    unique_vals = np.unique(W[W > 0])
    is_binary = len(unique_vals) <= 1

    if is_binary:
        # Assign excitatory/inhibitory signs per NODE (Dale's law):
        # each node is either excitatory or inhibitory
        n = W.shape[0]
        n_exc = int(n * exc_inh_ratio)
        signs = np.ones(n)
        inh_nodes = rng.choice(n, size=n - n_exc, replace=False)
        signs[inh_nodes] = -1
        # All outgoing connections from a node have the same sign
        W = W * signs[:, np.newaxis]
    else:
        # Weighted matrix: apply random sign to fraction of edges
        # to introduce inhibitory connections if all weights are positive
        if np.all(W >= 0):
            mask = W > 0
            sign_mask = rng.rand(*W.shape) > exc_inh_ratio
            W[mask & sign_mask] *= -1

    # Normalize spectral radius
    eigenvalues = linalg.eigvals(W)
    rho = np.max(np.abs(eigenvalues))

    if rho < 1e-10:
        return W

    W = W * (spectral_radius / rho)
    return W


def _run_reservoir(
    W: np.ndarray,
    inputs: np.ndarray,
    input_weights: np.ndarray,
    leak_rate: float = 1.0,
    bias: Optional[np.ndarray] = None,
) -> np.ndarray:
    """
    Drive reservoir with input signal and collect states.

    Parameters
    ----------
    W : (N, N) reservoir weight matrix
    inputs : (T,) or (T, n_inputs) input signal
    input_weights : (N, n_inputs) input-to-reservoir weights
    leak_rate : leaky integration rate (1.0 = no leak = standard ESN)
    bias : (N,) optional bias vector

    Returns
    -------
    states : (T, N) reservoir state matrix
    """
    N = W.shape[0]
    inputs = np.atleast_2d(inputs)
    if inputs.shape[0] == 1:
        inputs = inputs.T  # ensure (T, n_inputs)
    T = inputs.shape[0]

    states = np.zeros((T, N))
    x = np.zeros(N)

    if bias is None:
        bias = np.zeros(N)

    for t in range(T):
        u = input_weights @ inputs[t]
        x_new = np.tanh(W @ x + u + bias)
        x = (1 - leak_rate) * x + leak_rate * x_new
        states[t] = x

    return states


def _ridge_regression(X: np.ndarray, Y: np.ndarray, alpha: float = 1e-4) -> np.ndarray:
    """
    Solve Y = X @ W via ridge regression.
    Returns W of shape (n_features, n_targets).
    """
    # X: (T, n_features), Y: (T, n_targets)
    n = X.shape[1]
    W = linalg.solve(X.T @ X + alpha * np.eye(n), X.T @ Y, assume_a='pos')
    return W


def _r_squared(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Coefficient of determination, clipped to [0, 1]."""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot < 1e-12:
        return 0.0
    return float(np.clip(1 - ss_res / ss_tot, 0.0, 1.0))


def _effective_dimensionality(S: np.ndarray) -> float:
    """
    Effective dimensionality from singular values S.
    Uses the participation ratio: (sum s_i^2)^2 / sum s_i^4
    This equals 1 if one component dominates, N if all equal.
    """
    s2 = S ** 2
    s2 = s2[s2 > 1e-12]
    if len(s2) == 0:
        return 0.0
    return float((np.sum(s2)) ** 2 / np.sum(s2 ** 2))


def _spectral_entropy(S: np.ndarray) -> float:
    """
    Entropy of the normalized singular value spectrum.
    High entropy = rich, distributed dynamics. Low = low-dimensional.
    """
    s2 = S ** 2
    s2 = s2[s2 > 1e-12]
    if len(s2) == 0:
        return 0.0
    p = s2 / np.sum(s2)
    return float(-np.sum(p * np.log(p)))


def computational_capacity(
    A: np.ndarray,
    n_trials: int = 3,
    T_washout: int = 200,
    T_train: int = 2000,
    T_test: int = 500,
    spectral_radius: float = 0.95,
    input_scaling: float = 0.5,
    leak_rate: float = 1.0,
    max_delay: int = 30,
    max_nonlinear_delay: int = 15,
    ridge_alpha: float = 1e-4,
    seed: int = 42,
) -> dict:
    """
    Comprehensive computational capacity assessment of a network.

    Parameters
    ----------
    A : np.ndarray, shape (N, N)
        Adjacency or connectivity matrix. Can be weighted or binary,
        directed or undirected. Will be normalized to desired spectral radius.

    n_trials : int
        Number of independent random input trials to average over.
        More trials = more stable estimates, but slower.

    T_washout : int
        Initial timesteps to discard (let transients decay).

    T_train : int
        Timesteps for training the readout.

    T_test : int
        Timesteps for evaluating capacity (held-out).

    spectral_radius : float
        Target spectral radius for reservoir. 0.9 is standard;
        values near 1.0 probe edge-of-chaos dynamics.

    input_scaling : float
        Scaling of input weights. Small values keep dynamics in
        linear regime of tanh; larger values probe nonlinear regime.

    leak_rate : float
        Leaky integrator rate. 1.0 = standard ESN (no leak).
        Values < 1 slow down dynamics (longer timescales).

    max_delay : int
        Maximum lag for linear memory capacity evaluation.

    max_nonlinear_delay : int
        Maximum lag for nonlinear (quadratic) targets. Kept shorter
        than linear because nonlinear capacity decays faster.

    ridge_alpha : float
        Regularization for ridge regression readout.

    seed : int
        Random seed for reproducibility.

    Returns
    -------
    results : dict
        Comprehensive capacity profile with the following keys:

        ## Linear memory capacity
        'memory_capacity_total' : float
            Total linear MC = sum of R² across all lags.
            Theoretical maximum = N (number of nodes).
        'memory_capacity_profile' : np.ndarray, shape (max_delay,)
            R² at each lag k = 1, ..., max_delay.
        'memory_timescale' : float
            Characteristic timescale: lag at which MC drops to 1/e of peak.

        ## Nonlinear computation capacity
        'nonlinear_capacity_total' : float
            Total quadratic MC = sum of R² for x(t-k)² targets.
        'nonlinear_capacity_profile' : np.ndarray, shape (max_nonlinear_delay,)
            R² for each quadratic target.
        'cubic_capacity_total' : float
            Total cubic MC = sum of R² for x(t-k)³ targets.

        ## Cross-term capacity
        'cross_capacity_total' : float
            Total R² for x(t-i)*x(t-j) interaction targets.
            Measures ability to compute conjunctions of past inputs.

        ## Capacity balance
        'memory_nonlinear_ratio' : float
            MC_linear / (MC_linear + MC_nonlinear). Near 1 = memory-dominated
            (routing-like). Near 0 = transformation-dominated (diffusion-like).
        'total_capacity' : float
            Sum of linear + quadratic + cubic + cross capacities.

        ## State richness
        'state_dimensionality' : float
            Participation ratio of reservoir state singular values.
            Higher = richer dynamics exploiting more of the network.
        'state_entropy' : float
            Entropy of normalized singular value spectrum.
        'state_rank' : int
            Numerical rank of the state matrix (singular values > 1e-6).

        ## Input separation
        'separation_ratio' : float
            How well the reservoir separates distinct input streams.
            Measured as ratio of between-class to within-class state variance.

        ## Edge-of-chaos / stability
        'lyapunov_exponent' : float
            Estimated largest Lyapunov exponent from state perturbation.
            Negative = stable (ordered), near 0 = edge of chaos, positive = chaotic.

        ## Summary scores (for quick use in PCA/embedding)
        'capacity_vector' : np.ndarray, shape (8,)
            [memory_total, nonlinear_total, cubic_total, cross_total,
             memory_nonlinear_ratio, state_dimensionality, separation_ratio,
             lyapunov_exponent]
    """
    rng = np.random.RandomState(seed)
    N = A.shape[0]
    T_total = T_washout + T_train + T_test + max_delay

    # Prepare reservoir (signs assigned with rng for reproducibility)
    W = _prepare_reservoir_weights(A, spectral_radius=spectral_radius, rng=rng)

    # Accumulators across trials
    mc_profiles = []
    nl_profiles = []
    cb_profiles = []
    cross_totals = []
    state_dims = []
    state_ents = []
    state_ranks = []
    sep_ratios = []
    lyap_exps = []

    for trial in range(n_trials):
        trial_seed = seed + trial * 1000
        rng_trial = np.random.RandomState(trial_seed)

        # ---- Generate input and drive reservoir ----
        # Uniform input in [-1, 1] (standard for capacity measurement)
        u = rng_trial.uniform(-1, 1, size=T_total)

        # Input weights: all nodes receive input (standard for capacity measurement)
        # Drawn from uniform [-1, 1] and scaled by input_scaling
        w_in = rng_trial.uniform(-1, 1, size=N) * input_scaling
        w_in = w_in.reshape(-1, 1)

        # Small random bias to break symmetry
        bias = rng_trial.uniform(-0.1, 0.1, size=N)

        # Run reservoir
        states = _run_reservoir(W, u.reshape(-1, 1), w_in, leak_rate=leak_rate, bias=bias)

        # Discard washout
        states = states[T_washout:]
        u_valid = u[T_washout:]

        # For predicting u(t-k) from state(t), we need t >= max_delay
        # so all lags 1..max_delay are available as past inputs.
        # States usable for train+test start at index max_delay.
        S_usable = states[max_delay:]  # length = T_train + T_test
        S_train = S_usable[:T_train]
        S_test = S_usable[T_train:T_train + T_test]

        # ---- 1. LINEAR MEMORY CAPACITY ----
        mc_profile = np.zeros(max_delay)
        for k in range(1, max_delay + 1):
            # Target: input k steps before each state
            # S_train[j] = state at absolute time (max_delay + j)
            # target[j] = u_valid[max_delay + j - k]
            y_all = u_valid[max_delay - k : max_delay - k + T_train + T_test]
            y_train = y_all[:T_train]
            y_test = y_all[T_train:T_train + T_test]

            if len(y_train) < 10 or len(y_test) < 10:
                continue

            W_out = _ridge_regression(S_train, y_train.reshape(-1, 1), alpha=ridge_alpha)
            y_pred = S_test @ W_out
            mc_profile[k - 1] = _r_squared(y_test, y_pred.ravel())

        mc_profiles.append(mc_profile)

        # ---- 2. NONLINEAR (QUADRATIC) CAPACITY ----
        nl_profile = np.zeros(max_nonlinear_delay)
        for k in range(1, max_nonlinear_delay + 1):
            y_all = u_valid[max_delay - k : max_delay - k + T_train + T_test]
            y_nl = y_all ** 2

            y_train_nl = y_nl[:T_train]
            y_test_nl = y_nl[T_train:T_train + T_test]

            if len(y_train_nl) < 10 or len(y_test_nl) < 10:
                continue

            W_out = _ridge_regression(S_train, y_train_nl.reshape(-1, 1), alpha=ridge_alpha)
            y_pred = S_test @ W_out
            nl_profile[k - 1] = _r_squared(y_test_nl, y_pred.ravel())

        nl_profiles.append(nl_profile)

        # ---- 3. CUBIC CAPACITY ----
        cb_profile = np.zeros(max_nonlinear_delay)
        for k in range(1, max_nonlinear_delay + 1):
            y_all = u_valid[max_delay - k : max_delay - k + T_train + T_test]
            y_cb = y_all ** 3

            y_train_cb = y_cb[:T_train]
            y_test_cb = y_cb[T_train:T_train + T_test]

            if len(y_train_cb) < 10 or len(y_test_cb) < 10:
                continue

            W_out = _ridge_regression(S_train, y_train_cb.reshape(-1, 1), alpha=ridge_alpha)
            y_pred = S_test @ W_out
            cb_profile[k - 1] = _r_squared(y_test_cb, y_pred.ravel())

        cb_profiles.append(cb_profile)

        # ---- 4. CROSS-TERM CAPACITY ----
        cross_total = 0.0
        n_cross = 0
        for i in range(1, min(8, max_nonlinear_delay)):
            for j in range(i + 1, min(8, max_nonlinear_delay)):
                y_i = u_valid[max_delay - i : max_delay - i + T_train + T_test]
                y_j = u_valid[max_delay - j : max_delay - j + T_train + T_test]
                y_cross = y_i * y_j

                y_train_c = y_cross[:T_train]
                y_test_c = y_cross[T_train:T_train + T_test]

                if len(y_train_c) < 10 or len(y_test_c) < 10:
                    continue

                W_out = _ridge_regression(S_train, y_train_c.reshape(-1, 1), alpha=ridge_alpha)
                y_pred = S_test @ W_out
                cross_total += _r_squared(y_test_c, y_pred.ravel())
                n_cross += 1

        cross_totals.append(cross_total)

        # ---- 5. STATE RICHNESS ----
        # SVD of the full usable states for robust dimensionality estimate
        S_centered = S_usable - S_usable.mean(axis=0)
        try:
            _, S_vals, _ = linalg.svd(S_centered, full_matrices=False)
            state_dims.append(_effective_dimensionality(S_vals))
            state_ents.append(_spectral_entropy(S_vals))
            state_ranks.append(int(np.sum(S_vals > 1e-6 * S_vals[0])))
        except linalg.LinAlgError:
            state_dims.append(0.0)
            state_ents.append(0.0)
            state_ranks.append(0)

        # ---- 6. INPUT SEPARATION ----
        # Drive with two distinct input streams, measure state divergence
        u2 = rng_trial.uniform(-1, 1, size=T_washout + T_test)
        states_alt = _run_reservoir(W, u2.reshape(-1, 1), w_in, leak_rate=leak_rate, bias=bias)
        states_alt = states_alt[T_washout:]

        # Also re-run original input for fair comparison
        u1_seg = u[:T_washout + T_test]
        states_orig = _run_reservoir(W, u1_seg.reshape(-1, 1), w_in, leak_rate=leak_rate, bias=bias)
        states_orig = states_orig[T_washout:]

        min_t = min(len(states_orig), len(states_alt), T_test)
        states_orig = states_orig[:min_t]
        states_alt = states_alt[:min_t]

        # Between-stream distance vs within-stream variance
        between_var = np.mean(np.sum((states_orig - states_alt) ** 2, axis=1))
        within_var = 0.5 * (np.var(states_orig, axis=0).sum() +
                            np.var(states_alt, axis=0).sum())
        if within_var > 1e-12:
            sep_ratios.append(float(between_var / within_var))
        else:
            sep_ratios.append(0.0)

        # ---- 7. LYAPUNOV EXPONENT ESTIMATE ----
        # Perturb initial state slightly, track divergence
        u_lyap = rng_trial.uniform(-1, 1, size=T_washout + 200)

        states_base = _run_reservoir(W, u_lyap.reshape(-1, 1), w_in,
                                     leak_rate=leak_rate, bias=bias)

        # Perturbed run: nudge state at t=T_washout
        eps = 1e-8
        x_base = states_base[T_washout - 1].copy()
        x_pert = x_base + rng_trial.randn(N) * eps

        # Continue both from T_washout with same input
        n_lyap = 200
        divergences = np.zeros(n_lyap)
        x_b = x_base.copy()
        x_p = x_pert.copy()

        for t in range(n_lyap):
            idx = T_washout + t
            if idx >= len(u_lyap):
                break
            inp = w_in.ravel() * u_lyap[idx] + bias
            x_b = (1 - leak_rate) * x_b + leak_rate * np.tanh(W @ x_b + inp)
            x_p = (1 - leak_rate) * x_p + leak_rate * np.tanh(W @ x_p + inp)
            divergences[t] = np.log(np.linalg.norm(x_p - x_b) + 1e-20)

        # Linear fit to log-divergence gives Lyapunov exponent
        valid = np.isfinite(divergences) & (divergences > -40)
        if np.sum(valid) > 10:
            t_vals = np.arange(n_lyap)[valid]
            d_vals = divergences[valid]
            # Simple linear regression
            slope = np.polyfit(t_vals, d_vals, 1)[0]
            lyap_exps.append(float(slope))
        else:
            lyap_exps.append(0.0)

    # ---- AGGREGATE ACROSS TRIALS ----
    mc_profile = np.mean(mc_profiles, axis=0)
    nl_profile = np.mean(nl_profiles, axis=0)
    cb_profile = np.mean(cb_profiles, axis=0)

    mc_total = float(np.sum(mc_profile))
    nl_total = float(np.sum(nl_profile))
    cb_total = float(np.sum(cb_profile))
    cross_total = float(np.mean(cross_totals))

    total_cap = mc_total + nl_total + cb_total + cross_total

    # Memory timescale: find lag where MC drops to 1/e of peak
    if mc_profile.max() > 0:
        threshold = mc_profile.max() / np.e
        below = np.where(mc_profile < threshold)[0]
        memory_timescale = float(below[0] + 1) if len(below) > 0 else float(max_delay)
    else:
        memory_timescale = 0.0

    # Memory vs nonlinear balance
    denom = mc_total + nl_total
    mn_ratio = float(mc_total / denom) if denom > 1e-12 else 0.5

    state_dim = float(np.mean(state_dims))
    state_ent = float(np.mean(state_ents))
    state_rk = int(np.round(np.mean(state_ranks)))
    sep_ratio = float(np.mean(sep_ratios))
    lyap_exp = float(np.mean(lyap_exps))

    # ---- BUILD RESULTS ----
    results = {
        # Linear memory
        'memory_capacity_total': mc_total,
        'memory_capacity_profile': mc_profile,
        'memory_timescale': memory_timescale,
        # Nonlinear
        'nonlinear_capacity_total': nl_total,
        'nonlinear_capacity_profile': nl_profile,
        'cubic_capacity_total': cb_total,
        'cubic_capacity_profile': cb_profile,
        # Cross-terms
        'cross_capacity_total': cross_total,
        # Balance
        'memory_nonlinear_ratio': mn_ratio,
        'total_capacity': total_cap,
        # State richness
        'state_dimensionality': state_dim,
        'state_entropy': state_ent,
        'state_rank': state_rk,
        # Separation
        'separation_ratio': sep_ratio,
        # Stability
        'lyapunov_exponent': lyap_exp,
        # Summary vector for embedding / PCA (8 features)
        'capacity_vector': np.array([
            mc_total,
            nl_total,
            cb_total,
            cross_total,
            mn_ratio,
            state_dim,
            sep_ratio,
            lyap_exp,
        ]),
    }

    return results


# ---- BATCH PROCESSING UTILITIES ----

def batch_computational_capacity(
    adjacency_matrices: list,
    labels: Optional[list] = None,
    n_trials: int = 3,
    seed: int = 42,
    verbose: bool = True,
    **kwargs,
) -> dict:
    """
    Evaluate computational capacity for a list of networks.

    Parameters
    ----------
    adjacency_matrices : list of np.ndarray
        Each element is an (N, N) adjacency matrix.
    labels : list of str, optional
        Labels for each network (e.g., 'human_001', 'mami_mouse', 'synth_42').
    n_trials : int
        Trials per network.
    seed : int
        Base seed.
    verbose : bool
        Print progress.

    Returns
    -------
    batch_results : dict with keys:
        'capacity_matrix' : np.ndarray (n_networks, 8)
            capacity_vector for each network, ready for PCA.
        'scalar_results' : dict of np.ndarray
            Each scalar metric as an array of length n_networks.
        'individual_results' : list of dict
            Full results for each network.
    """
    import time

    n = len(adjacency_matrices)
    if labels is None:
        labels = [f'network_{i}' for i in range(n)]

    individual = []
    capacity_matrix = np.zeros((n, 8))

    scalar_keys = [
        'memory_capacity_total', 'nonlinear_capacity_total',
        'cubic_capacity_total', 'cross_capacity_total',
        'memory_nonlinear_ratio', 'total_capacity',
        'state_dimensionality', 'state_entropy', 'state_rank',
        'separation_ratio', 'lyapunov_exponent', 'memory_timescale',
    ]
    scalars = {k: np.zeros(n) for k in scalar_keys}

    t0 = time.time()
    for i, A in enumerate(adjacency_matrices):
        res = computational_capacity(A, n_trials=n_trials, seed=seed + i, **kwargs)
        individual.append(res)
        capacity_matrix[i] = res['capacity_vector']
        for k in scalar_keys:
            scalars[k][i] = res[k]

        if verbose and (i + 1) % 100 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            eta = (n - i - 1) / rate
            print(f"  [{i+1}/{n}] {rate:.1f} nets/sec, ETA: {eta/60:.1f} min")

    return {
        'capacity_matrix': capacity_matrix,
        'scalar_results': scalars,
        'individual_results': individual,
        'labels': labels,
    }


# ---- QUICK VALIDATION ----

# if __name__ == '__main__':
np.random.seed(42)
N = 100

print("=" * 60)
print("Validating computational_capacity on test networks")
print("=" * 60)

# 1. Random Erdos-Renyi network
A_random = (np.random.rand(N, N) < 0.1).astype(float)
A_random = np.triu(A_random, 1)
A_random = A_random + A_random.T
print("\n--- Random ER network (p=0.1) ---")
res_rand = computational_capacity(A_random, n_trials=2, T_train=1000, T_test=300)
print(f"  Memory capacity:     {res_rand['memory_capacity_total']:.2f}")
print(f"  Nonlinear capacity:  {res_rand['nonlinear_capacity_total']:.2f}")
print(f"  Cubic capacity:      {res_rand['cubic_capacity_total']:.2f}")
print(f"  Cross capacity:      {res_rand['cross_capacity_total']:.2f}")
print(f"  Total capacity:      {res_rand['total_capacity']:.2f}")
print(f"  Memory/NL ratio:     {res_rand['memory_nonlinear_ratio']:.3f}")
print(f"  Memory timescale:    {res_rand['memory_timescale']:.1f}")
print(f"  State dimensionality:{res_rand['state_dimensionality']:.1f}")
print(f"  State entropy:       {res_rand['state_entropy']:.2f}")
print(f"  Separation ratio:    {res_rand['separation_ratio']:.2f}")
print(f"  Lyapunov exponent:   {res_rand['lyapunov_exponent']:.4f}")

# 2. Ring lattice (high regularity, long memory)
A_ring = np.zeros((N, N))
for i in range(N):
    for k in [1, 2, 3]:
        A_ring[i, (i + k) % N] = 1
        A_ring[i, (i - k) % N] = 1
print("\n--- Ring lattice (k=3) ---")
res_ring = computational_capacity(A_ring, n_trials=2, T_train=1000, T_test=300)
print(f"  Memory capacity:     {res_ring['memory_capacity_total']:.2f}")
print(f"  Nonlinear capacity:  {res_ring['nonlinear_capacity_total']:.2f}")
print(f"  Memory/NL ratio:     {res_ring['memory_nonlinear_ratio']:.3f}")
print(f"  State dimensionality:{res_ring['state_dimensionality']:.1f}")
print(f"  Lyapunov exponent:   {res_ring['lyapunov_exponent']:.4f}")

# 3. Modular network (4 modules)
A_mod = np.zeros((N, N))
mod_size = N // 4
for m in range(4):
    start = m * mod_size
    end = start + mod_size
    block = (np.random.rand(mod_size, mod_size) < 0.3).astype(float)
    block = np.triu(block, 1)
    block = block + block.T
    A_mod[start:end, start:end] = block
# Sparse between-module connections
for i in range(N):
    for j in range(i + 1, N):
        if i // mod_size != j // mod_size and np.random.rand() < 0.02:
            A_mod[i, j] = 1
            A_mod[j, i] = 1
print("\n--- Modular network (4 modules) ---")
res_mod = computational_capacity(A_mod, n_trials=2, T_train=1000, T_test=300)
print(f"  Memory capacity:     {res_mod['memory_capacity_total']:.2f}")
print(f"  Nonlinear capacity:  {res_mod['nonlinear_capacity_total']:.2f}")
print(f"  Memory/NL ratio:     {res_mod['memory_nonlinear_ratio']:.3f}")
print(f"  State dimensionality:{res_mod['state_dimensionality']:.1f}")
print(f"  Lyapunov exponent:   {res_mod['lyapunov_exponent']:.4f}")

print("\n" + "=" * 60)
print("Capacity vectors (for embedding):")
print(f"  Random:  {res_rand['capacity_vector']}")
print(f"  Ring:    {res_ring['capacity_vector']}")
print(f"  Modular: {res_mod['capacity_vector']}")
print("=" * 60)

Validating computational_capacity on test networks

--- Random ER network (p=0.1) ---
  Memory capacity:     8.52
  Nonlinear capacity:  3.42
  Cubic capacity:      7.23
  Cross capacity:      7.66
  Total capacity:      26.83
  Memory/NL ratio:     0.714
  Memory timescale:    10.0
  State dimensionality:1.4
  State entropy:       0.55
  Separation ratio:    2.05
  Lyapunov exponent:   -0.0985

--- Ring lattice (k=3) ---


/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_42870/2008174171.py:121: RuntimeWarning: divide by zero encountered in matmul
  x_new = np.tanh(W @ x + u + bias)
/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_42870/2008174171.py:121: RuntimeWarning: overflow encountered in matmul
  x_new = np.tanh(W @ x + u + bias)
/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_42870/2008174171.py:121: RuntimeWarning: invalid value encountered in matmul
  x_new = np.tanh(W @ x + u + bias)
/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_42870/2008174171.py:135: RuntimeWarning: divide by zero encountered in matmul
  W = linalg.solve(X.T @ X + alpha * np.eye(n), X.T @ Y, assume_a='pos')
/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_42870/2008174171.py:135: RuntimeWarning: overflow encountered in matmul
  W = linalg.solve(X.T @ X + alpha * np.eye(n), X.T @ Y, assume_a='pos')
/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_42870/200817417

  Memory capacity:     7.56
  Nonlinear capacity:  3.37
  Memory/NL ratio:     0.692
  State dimensionality:1.5
  Lyapunov exponent:   -0.1177

--- Modular network (4 modules) ---
  Memory capacity:     8.73
  Nonlinear capacity:  3.59
  Memory/NL ratio:     0.708
  State dimensionality:1.7
  Lyapunov exponent:   -0.0993

Capacity vectors (for embedding):
  Random:  [ 8.51716999  3.41613368  7.22793845  7.66398097  0.7137311   1.37702212
  2.05296769 -0.0985287 ]
  Ring:    [ 7.55853292  3.37006303  6.29717099  5.05611107  0.69162891  1.48901859
  2.05197683 -0.1177081 ]
  Modular: [ 8.72551022  3.59068113  7.29659529  7.25498961  0.70845848  1.71729531
  2.07899562 -0.09933679]


In [8]:
res_rand

{'memory_capacity_total': 8.517169992495042,
 'memory_capacity_profile': array([0.99999194, 0.99994713, 0.99973107, 0.99858873, 0.99545392,
        0.98427422, 0.93990337, 0.82246713, 0.39626687, 0.18168842,
        0.10340862, 0.05541956, 0.03025875, 0.00106421, 0.00261483,
        0.00490831, 0.00118291, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ]),
 'memory_timescale': 10.0,
 'nonlinear_capacity_total': 3.416133677752688,
 'nonlinear_capacity_profile': array([0.99485568, 0.96660824, 0.84708868, 0.46460927, 0.12001887,
        0.01387449, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.00656941, 0.        , 0.        , 0.00250902]),
 'cubic_capacity_total': 7.227938454330771,
 'cubic_capacity_profile': array([0.99442092, 0.97376086, 0.90973129, 0.83903973, 0.81996436,
        0.8014648 , 0.75159771, 0.64862333, 0.29781256, 0.12195518,
 

In [9]:
"""
Comprehensive computational capacity assessment of network architectures
via Echo State Network (ESN) reservoir computing.

Measures multiple facets of computation:
  - Linear memory capacity (faithful recall of past inputs)
  - Nonlinear capacity (quadratic/cubic transformations of past inputs)
  - Cross-term capacity (interaction between inputs at different lags)
  - Input separation (discrimination between distinct input streams)
  - State richness (dimensionality and entropy of reservoir dynamics)
  - Edge-of-chaos proximity (Lyapunov exponent estimate)

Compatible with any (N, N) adjacency matrix. Designed for batch evaluation
across tens of thousands of networks.

References:
  - Jaeger (2001) - Echo state networks
  - Büsing et al. (2010) - Connectivity, dynamics, and memory in RC
  - Dambre et al. (2012) - Information processing capacity of dynamical systems
  - Suárez et al. (2024) - Connectome-based reservoir computing (conn2res)
"""

import numpy as np
from scipy import linalg
from typing import Optional


def _prepare_reservoir_weights(
    A: np.ndarray,
    spectral_radius: float = 0.95,
    exc_inh_ratio: float = 0.8,
    rng: np.random.RandomState = None,
) -> np.ndarray:
    """
    Normalize adjacency matrix for ESN use.

    For binary matrices: assigns random excitatory/inhibitory signs (Dale's law)
    to create heterogeneous dynamics. For weighted matrices: preserves relative
    weight structure.

    Always rescales to target spectral_radius.
    """
    if rng is None:
        rng = np.random.RandomState(0)

    W = A.copy().astype(np.float64)
    np.fill_diagonal(W, 0)

    if np.allclose(W, 0):
        return W

    # Check if binary (only 0s and 1s)
    unique_vals = np.unique(W[W > 0])
    is_binary = len(unique_vals) <= 1

    if is_binary:
        # Assign excitatory/inhibitory signs per NODE (Dale's law):
        # each node is either excitatory or inhibitory
        n = W.shape[0]
        n_exc = int(n * exc_inh_ratio)
        signs = np.ones(n)
        inh_nodes = rng.choice(n, size=n - n_exc, replace=False)
        signs[inh_nodes] = -1
        # All outgoing connections from a node have the same sign
        W = W * signs[:, np.newaxis]
    else:
        # Weighted matrix: apply random sign to fraction of edges
        # to introduce inhibitory connections if all weights are positive
        if np.all(W >= 0):
            mask = W > 0
            sign_mask = rng.rand(*W.shape) > exc_inh_ratio
            W[mask & sign_mask] *= -1

    # Normalize spectral radius
    eigenvalues = linalg.eigvals(W)
    rho = np.max(np.abs(eigenvalues))

    if rho < 1e-10:
        return W

    W = W * (spectral_radius / rho)
    return W


def _run_reservoir(
    W: np.ndarray,
    inputs: np.ndarray,
    input_weights: np.ndarray,
    leak_rate: float = 1.0,
    bias: Optional[np.ndarray] = None,
) -> np.ndarray:
    """
    Drive reservoir with input signal and collect states.

    Parameters
    ----------
    W : (N, N) reservoir weight matrix
    inputs : (T,) or (T, n_inputs) input signal
    input_weights : (N, n_inputs) input-to-reservoir weights
    leak_rate : leaky integration rate (1.0 = no leak = standard ESN)
    bias : (N,) optional bias vector

    Returns
    -------
    states : (T, N) reservoir state matrix
    """
    N = W.shape[0]
    inputs = np.atleast_2d(inputs)
    if inputs.shape[0] == 1:
        inputs = inputs.T  # ensure (T, n_inputs)
    T = inputs.shape[0]

    states = np.zeros((T, N))
    x = np.zeros(N)

    if bias is None:
        bias = np.zeros(N)

    for t in range(T):
        u = input_weights @ inputs[t]
        x_new = np.tanh(W @ x + u + bias)
        x = (1 - leak_rate) * x + leak_rate * x_new
        states[t] = x

    return states


def _ridge_regression(X: np.ndarray, Y: np.ndarray, alpha: float = 1e-4) -> np.ndarray:
    """
    Solve Y = X @ W via ridge regression.
    Returns W of shape (n_features, n_targets).
    """
    # X: (T, n_features), Y: (T, n_targets)
    n = X.shape[1]
    W = linalg.solve(X.T @ X + alpha * np.eye(n), X.T @ Y, assume_a='pos')
    return W


def _r_squared(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Coefficient of determination, clipped to [0, 1]."""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot < 1e-12:
        return 0.0
    return float(np.clip(1 - ss_res / ss_tot, 0.0, 1.0))


def _effective_dimensionality(S: np.ndarray) -> float:
    """
    Effective dimensionality from singular values S.
    Uses the participation ratio: (sum s_i^2)^2 / sum s_i^4
    This equals 1 if one component dominates, N if all equal.
    """
    s2 = S ** 2
    s2 = s2[s2 > 1e-12]
    if len(s2) == 0:
        return 0.0
    return float((np.sum(s2)) ** 2 / np.sum(s2 ** 2))


def _spectral_entropy(S: np.ndarray) -> float:
    """
    Entropy of the normalized singular value spectrum.
    High entropy = rich, distributed dynamics. Low = low-dimensional.
    """
    s2 = S ** 2
    s2 = s2[s2 > 1e-12]
    if len(s2) == 0:
        return 0.0
    p = s2 / np.sum(s2)
    return float(-np.sum(p * np.log(p)))


def computational_capacity(
    A: np.ndarray,
    n_trials: int = 3,
    T_washout: int = 200,
    T_train: int = 2000,
    T_test: int = 500,
    spectral_radius: float = 0.95,
    input_scaling: float = 0.5,
    leak_rate: float = 1.0,
    max_delay: int = 30,
    max_nonlinear_delay: int = 15,
    ridge_alpha: float = 1e-4,
    seed: int = 42,
) -> dict:
    """
    Comprehensive computational capacity assessment of a network.

    Parameters
    ----------
    A : np.ndarray, shape (N, N)
        Adjacency or connectivity matrix. Can be weighted or binary,
        directed or undirected. Will be normalized to desired spectral radius.

    n_trials : int
        Number of independent random input trials to average over.
        More trials = more stable estimates, but slower.

    T_washout : int
        Initial timesteps to discard (let transients decay).

    T_train : int
        Timesteps for training the readout.

    T_test : int
        Timesteps for evaluating capacity (held-out).

    spectral_radius : float
        Target spectral radius for reservoir. 0.9 is standard;
        values near 1.0 probe edge-of-chaos dynamics.

    input_scaling : float
        Scaling of input weights. Small values keep dynamics in
        linear regime of tanh; larger values probe nonlinear regime.

    leak_rate : float
        Leaky integrator rate. 1.0 = standard ESN (no leak).
        Values < 1 slow down dynamics (longer timescales).

    max_delay : int
        Maximum lag for linear memory capacity evaluation.

    max_nonlinear_delay : int
        Maximum lag for nonlinear (quadratic) targets. Kept shorter
        than linear because nonlinear capacity decays faster.

    ridge_alpha : float
        Regularization for ridge regression readout.

    seed : int
        Random seed for reproducibility.

    Returns
    -------
    results : dict
        Comprehensive capacity profile with the following keys:

        ## Linear memory capacity
        'memory_capacity_total' : float
            Total linear MC = sum of R² across all lags.
            Theoretical maximum = N (number of nodes).
        'memory_capacity_profile' : np.ndarray, shape (max_delay,)
            R² at each lag k = 1, ..., max_delay.
        'memory_timescale' : float
            Characteristic timescale: lag at which MC drops to 1/e of peak.

        ## Nonlinear computation capacity
        'nonlinear_capacity_total' : float
            Total quadratic MC = sum of R² for x(t-k)² targets.
        'nonlinear_capacity_profile' : np.ndarray, shape (max_nonlinear_delay,)
            R² for each quadratic target.
        'cubic_capacity_total' : float
            Total cubic MC = sum of R² for x(t-k)³ targets.

        ## Cross-term capacity
        'cross_capacity_total' : float
            Total R² for x(t-i)*x(t-j) interaction targets.
            Measures ability to compute conjunctions of past inputs.

        ## Capacity balance
        'memory_nonlinear_ratio' : float
            MC_linear / (MC_linear + MC_nonlinear). Near 1 = memory-dominated
            (routing-like). Near 0 = transformation-dominated (diffusion-like).
        'total_capacity' : float
            Sum of linear + quadratic + cubic + cross capacities.

        ## State richness
        'state_dimensionality' : float
            Participation ratio of reservoir state singular values.
            Higher = richer dynamics exploiting more of the network.
        'state_entropy' : float
            Entropy of normalized singular value spectrum.
        'state_rank' : int
            Numerical rank of the state matrix (singular values > 1e-6).

        ## Input separation
        'separation_ratio' : float
            How well the reservoir separates distinct input streams.
            Measured as ratio of between-class to within-class state variance.

        ## Edge-of-chaos / stability
        'lyapunov_exponent' : float
            Estimated largest Lyapunov exponent from state perturbation.
            Negative = stable (ordered), near 0 = edge of chaos, positive = chaotic.

        ## Summary scores (for quick use in PCA/embedding)
        'capacity_vector' : np.ndarray, shape (8,)
            [memory_total, nonlinear_total, cubic_total, cross_total,
             memory_nonlinear_ratio, state_dimensionality, separation_ratio,
             lyapunov_exponent]
    """
    rng = np.random.RandomState(seed)
    N = A.shape[0]
    T_total = T_washout + T_train + T_test + max_delay

    # Prepare reservoir (signs assigned with rng for reproducibility)
    W = _prepare_reservoir_weights(A, spectral_radius=spectral_radius, rng=rng)

    # Accumulators across trials
    mc_profiles = []
    nl_profiles = []
    cb_profiles = []
    cross_totals = []
    state_dims = []
    state_ents = []
    state_ranks = []
    sep_ratios = []
    lyap_exps = []

    for trial in range(n_trials):
        trial_seed = seed + trial * 1000
        rng_trial = np.random.RandomState(trial_seed)

        # ---- Generate input and drive reservoir ----
        # Uniform input in [-1, 1] (standard for capacity measurement)
        u = rng_trial.uniform(-1, 1, size=T_total)

        # Input weights: all nodes receive input (standard for capacity measurement)
        # Drawn from uniform [-1, 1] and scaled by input_scaling
        w_in = rng_trial.uniform(-1, 1, size=N) * input_scaling
        w_in = w_in.reshape(-1, 1)

        # Small random bias to break symmetry
        bias = rng_trial.uniform(-0.1, 0.1, size=N)

        # Run reservoir
        states = _run_reservoir(W, u.reshape(-1, 1), w_in, leak_rate=leak_rate, bias=bias)

        # Discard washout
        states = states[T_washout:]
        u_valid = u[T_washout:]

        # For predicting u(t-k) from state(t), we need t >= max_delay
        # so all lags 1..max_delay are available as past inputs.
        # States usable for train+test start at index max_delay.
        S_usable = states[max_delay:]  # length = T_train + T_test
        S_train = S_usable[:T_train]
        S_test = S_usable[T_train:T_train + T_test]

        # ---- 1. LINEAR MEMORY CAPACITY ----
        mc_profile = np.zeros(max_delay)
        for k in range(1, max_delay + 1):
            # Target: input k steps before each state
            # S_train[j] = state at absolute time (max_delay + j)
            # target[j] = u_valid[max_delay + j - k]
            y_all = u_valid[max_delay - k : max_delay - k + T_train + T_test]
            y_train = y_all[:T_train]
            y_test = y_all[T_train:T_train + T_test]

            if len(y_train) < 10 or len(y_test) < 10:
                continue

            W_out = _ridge_regression(S_train, y_train.reshape(-1, 1), alpha=ridge_alpha)
            y_pred = S_test @ W_out
            mc_profile[k - 1] = _r_squared(y_test, y_pred.ravel())

        mc_profiles.append(mc_profile)

        # ---- 2. NONLINEAR (QUADRATIC) CAPACITY ----
        nl_profile = np.zeros(max_nonlinear_delay)
        for k in range(1, max_nonlinear_delay + 1):
            y_all = u_valid[max_delay - k : max_delay - k + T_train + T_test]
            y_nl = y_all ** 2

            y_train_nl = y_nl[:T_train]
            y_test_nl = y_nl[T_train:T_train + T_test]

            if len(y_train_nl) < 10 or len(y_test_nl) < 10:
                continue

            W_out = _ridge_regression(S_train, y_train_nl.reshape(-1, 1), alpha=ridge_alpha)
            y_pred = S_test @ W_out
            nl_profile[k - 1] = _r_squared(y_test_nl, y_pred.ravel())

        nl_profiles.append(nl_profile)

        # ---- 3. CUBIC CAPACITY ----
        cb_profile = np.zeros(max_nonlinear_delay)
        for k in range(1, max_nonlinear_delay + 1):
            y_all = u_valid[max_delay - k : max_delay - k + T_train + T_test]
            y_cb = y_all ** 3

            y_train_cb = y_cb[:T_train]
            y_test_cb = y_cb[T_train:T_train + T_test]

            if len(y_train_cb) < 10 or len(y_test_cb) < 10:
                continue

            W_out = _ridge_regression(S_train, y_train_cb.reshape(-1, 1), alpha=ridge_alpha)
            y_pred = S_test @ W_out
            cb_profile[k - 1] = _r_squared(y_test_cb, y_pred.ravel())

        cb_profiles.append(cb_profile)

        # ---- 4. CROSS-TERM CAPACITY ----
        cross_total = 0.0
        n_cross = 0
        for i in range(1, min(8, max_nonlinear_delay)):
            for j in range(i + 1, min(8, max_nonlinear_delay)):
                y_i = u_valid[max_delay - i : max_delay - i + T_train + T_test]
                y_j = u_valid[max_delay - j : max_delay - j + T_train + T_test]
                y_cross = y_i * y_j

                y_train_c = y_cross[:T_train]
                y_test_c = y_cross[T_train:T_train + T_test]

                if len(y_train_c) < 10 or len(y_test_c) < 10:
                    continue

                W_out = _ridge_regression(S_train, y_train_c.reshape(-1, 1), alpha=ridge_alpha)
                y_pred = S_test @ W_out
                cross_total += _r_squared(y_test_c, y_pred.ravel())
                n_cross += 1

        cross_totals.append(cross_total)

        # ---- 5. STATE RICHNESS ----
        # SVD of the full usable states for robust dimensionality estimate
        S_centered = S_usable - S_usable.mean(axis=0)
        try:
            _, S_vals, _ = linalg.svd(S_centered, full_matrices=False)
            state_dims.append(_effective_dimensionality(S_vals))
            state_ents.append(_spectral_entropy(S_vals))
            state_ranks.append(int(np.sum(S_vals > 1e-6 * S_vals[0])))
        except linalg.LinAlgError:
            state_dims.append(0.0)
            state_ents.append(0.0)
            state_ranks.append(0)

        # ---- 6. INPUT SEPARATION ----
        # Drive with two distinct input streams, measure state divergence
        u2 = rng_trial.uniform(-1, 1, size=T_washout + T_test)
        states_alt = _run_reservoir(W, u2.reshape(-1, 1), w_in, leak_rate=leak_rate, bias=bias)
        states_alt = states_alt[T_washout:]

        # Also re-run original input for fair comparison
        u1_seg = u[:T_washout + T_test]
        states_orig = _run_reservoir(W, u1_seg.reshape(-1, 1), w_in, leak_rate=leak_rate, bias=bias)
        states_orig = states_orig[T_washout:]

        min_t = min(len(states_orig), len(states_alt), T_test)
        states_orig = states_orig[:min_t]
        states_alt = states_alt[:min_t]

        # Between-stream distance vs within-stream variance
        between_var = np.mean(np.sum((states_orig - states_alt) ** 2, axis=1))
        within_var = 0.5 * (np.var(states_orig, axis=0).sum() +
                            np.var(states_alt, axis=0).sum())
        if within_var > 1e-12:
            sep_ratios.append(float(between_var / within_var))
        else:
            sep_ratios.append(0.0)

        # ---- 7. LYAPUNOV EXPONENT ESTIMATE ----
        # Perturb initial state slightly, track divergence
        u_lyap = rng_trial.uniform(-1, 1, size=T_washout + 200)

        states_base = _run_reservoir(W, u_lyap.reshape(-1, 1), w_in,
                                     leak_rate=leak_rate, bias=bias)

        # Perturbed run: nudge state at t=T_washout
        eps = 1e-8
        x_base = states_base[T_washout - 1].copy()
        x_pert = x_base + rng_trial.randn(N) * eps

        # Continue both from T_washout with same input
        n_lyap = 200
        divergences = np.zeros(n_lyap)
        x_b = x_base.copy()
        x_p = x_pert.copy()

        for t in range(n_lyap):
            idx = T_washout + t
            if idx >= len(u_lyap):
                break
            inp = w_in.ravel() * u_lyap[idx] + bias
            x_b = (1 - leak_rate) * x_b + leak_rate * np.tanh(W @ x_b + inp)
            x_p = (1 - leak_rate) * x_p + leak_rate * np.tanh(W @ x_p + inp)
            divergences[t] = np.log(np.linalg.norm(x_p - x_b) + 1e-20)

        # Linear fit to log-divergence gives Lyapunov exponent
        valid = np.isfinite(divergences) & (divergences > -40)
        if np.sum(valid) > 10:
            t_vals = np.arange(n_lyap)[valid]
            d_vals = divergences[valid]
            # Simple linear regression
            slope = np.polyfit(t_vals, d_vals, 1)[0]
            lyap_exps.append(float(slope))
        else:
            lyap_exps.append(0.0)

    # ---- AGGREGATE ACROSS TRIALS ----
    mc_profile = np.mean(mc_profiles, axis=0)
    nl_profile = np.mean(nl_profiles, axis=0)
    cb_profile = np.mean(cb_profiles, axis=0)

    mc_total = float(np.sum(mc_profile))
    nl_total = float(np.sum(nl_profile))
    cb_total = float(np.sum(cb_profile))
    cross_total = float(np.mean(cross_totals))

    total_cap = mc_total + nl_total + cb_total + cross_total

    # Memory timescale: find lag where MC drops to 1/e of peak
    if mc_profile.max() > 0:
        threshold = mc_profile.max() / np.e
        below = np.where(mc_profile < threshold)[0]
        memory_timescale = float(below[0] + 1) if len(below) > 0 else float(max_delay)
    else:
        memory_timescale = 0.0

    # Memory vs nonlinear balance
    denom = mc_total + nl_total
    mn_ratio = float(mc_total / denom) if denom > 1e-12 else 0.5

    state_dim = float(np.mean(state_dims))
    state_ent = float(np.mean(state_ents))
    state_rk = int(np.round(np.mean(state_ranks)))
    sep_ratio = float(np.mean(sep_ratios))
    lyap_exp = float(np.mean(lyap_exps))

    # ---- BUILD RESULTS ----
    results = {
        # Linear memory
        'memory_capacity_total': mc_total,
        # 'memory_capacity_profile': mc_profile,
        'memory_timescale': memory_timescale,
        # Nonlinear
        'nonlinear_capacity_total': nl_total,
        # 'nonlinear_capacity_profile': nl_profile,
        'cubic_capacity_total': cb_total,
        # 'cubic_capacity_profile': cb_profile,
        # Cross-terms
        'cross_capacity_total': cross_total,
        # Balance
        'memory_nonlinear_ratio': mn_ratio,
        'total_capacity': total_cap,
        # State richness
        'state_dimensionality': state_dim,
        'state_entropy': state_ent,
        'state_rank': state_rk,
        # Separation
        'separation_ratio': sep_ratio,
        # Stability
        'lyapunov_exponent': lyap_exp,
        # Summary vector for embedding / PCA (8 features)
        # 'capacity_vector': np.array([
        #     mc_total,
        #     nl_total,
        #     cb_total,
        #     cross_total,
        #     mn_ratio,
        #     state_dim,
        #     sep_ratio,
        #     lyap_exp,
        # ]),
    }

    return results


# # ---- BATCH PROCESSING UTILITIES ----

# def batch_computational_capacity(
#     adjacency_matrices: list,
#     labels: Optional[list] = None,
#     n_trials: int = 3,
#     seed: int = 42,
#     verbose: bool = True,
#     **kwargs,
# ) -> dict:
#     """
#     Evaluate computational capacity for a list of networks.

#     Parameters
#     ----------
#     adjacency_matrices : list of np.ndarray
#         Each element is an (N, N) adjacency matrix.
#     labels : list of str, optional
#         Labels for each network (e.g., 'human_001', 'mami_mouse', 'synth_42').
#     n_trials : int
#         Trials per network.
#     seed : int
#         Base seed.
#     verbose : bool
#         Print progress.

#     Returns
#     -------
#     batch_results : dict with keys:
#         'capacity_matrix' : np.ndarray (n_networks, 8)
#             capacity_vector for each network, ready for PCA.
#         'scalar_results' : dict of np.ndarray
#             Each scalar metric as an array of length n_networks.
#         'individual_results' : list of dict
#             Full results for each network.
#     """
#     import time

#     n = len(adjacency_matrices)
#     if labels is None:
#         labels = [f'network_{i}' for i in range(n)]

#     individual = []
#     capacity_matrix = np.zeros((n, 8))

#     scalar_keys = [
#         'memory_capacity_total', 'nonlinear_capacity_total',
#         'cubic_capacity_total', 'cross_capacity_total',
#         'memory_nonlinear_ratio', 'total_capacity',
#         'state_dimensionality', 'state_entropy', 'state_rank',
#         'separation_ratio', 'lyapunov_exponent', 'memory_timescale',
#     ]
#     scalars = {k: np.zeros(n) for k in scalar_keys}

#     t0 = time.time()
#     for i, A in enumerate(adjacency_matrices):
#         res = computational_capacity(A, n_trials=n_trials, seed=seed + i, **kwargs)
#         individual.append(res)
#         capacity_matrix[i] = res['capacity_vector']
#         for k in scalar_keys:
#             scalars[k][i] = res[k]

#         if verbose and (i + 1) % 100 == 0:
#             elapsed = time.time() - t0
#             rate = (i + 1) / elapsed
#             eta = (n - i - 1) / rate
#             print(f"  [{i+1}/{n}] {rate:.1f} nets/sec, ETA: {eta/60:.1f} min")

#     return {
#         'capacity_matrix': capacity_matrix,
#         'scalar_results': scalars,
#         'individual_results': individual,
#         'labels': labels,
#     }


# ---- QUICK VALIDATION ----

# if __name__ == '__main__':
np.random.seed(42)
N = 100

print("=" * 60)
print("Validating computational_capacity on test networks")
print("=" * 60)

# 1. Random Erdos-Renyi network
A_random = (np.random.rand(N, N) < 0.1).astype(float)
A_random = np.triu(A_random, 1)
A_random = A_random + A_random.T
print("\n--- Random ER network (p=0.1) ---")
res_rand = computational_capacity(A_random, n_trials=2, T_train=1000, T_test=300)
# print(f"  Memory capacity:     {res_rand['memory_capacity_total']:.2f}")
# print(f"  Nonlinear capacity:  {res_rand['nonlinear_capacity_total']:.2f}")
# print(f"  Cubic capacity:      {res_rand['cubic_capacity_total']:.2f}")
# print(f"  Cross capacity:      {res_rand['cross_capacity_total']:.2f}")
# print(f"  Total capacity:      {res_rand['total_capacity']:.2f}")
# print(f"  Memory/NL ratio:     {res_rand['memory_nonlinear_ratio']:.3f}")
# print(f"  Memory timescale:    {res_rand['memory_timescale']:.1f}")
# print(f"  State dimensionality:{res_rand['state_dimensionality']:.1f}")
# print(f"  State entropy:       {res_rand['state_entropy']:.2f}")
# print(f"  Separation ratio:    {res_rand['separation_ratio']:.2f}")
# print(f"  Lyapunov exponent:   {res_rand['lyapunov_exponent']:.4f}")
res_rand 
# 2. Ring lattice (high regularity, long memory)
# A_ring = np.zeros((N, N))
# for i in range(N):
#     for k in [1, 2, 3]:
#         A_ring[i, (i + k) % N] = 1
#         A_ring[i, (i - k) % N] = 1
# print("\n--- Ring lattice (k=3) ---")
# res_ring = computational_capacity(A_ring, n_trials=2, T_train=1000, T_test=300)
# print(f"  Memory capacity:     {res_ring['memory_capacity_total']:.2f}")
# print(f"  Nonlinear capacity:  {res_ring['nonlinear_capacity_total']:.2f}")
# print(f"  Memory/NL ratio:     {res_ring['memory_nonlinear_ratio']:.3f}")
# print(f"  State dimensionality:{res_ring['state_dimensionality']:.1f}")
# print(f"  Lyapunov exponent:   {res_ring['lyapunov_exponent']:.4f}")

# # 3. Modular network (4 modules)
# A_mod = np.zeros((N, N))
# mod_size = N // 4
# for m in range(4):
#     start = m * mod_size
#     end = start + mod_size
#     block = (np.random.rand(mod_size, mod_size) < 0.3).astype(float)
#     block = np.triu(block, 1)
#     block = block + block.T
#     A_mod[start:end, start:end] = block
# # Sparse between-module connections
# for i in range(N):
#     for j in range(i + 1, N):
#         if i // mod_size != j // mod_size and np.random.rand() < 0.02:
#             A_mod[i, j] = 1
#             A_mod[j, i] = 1
# print("\n--- Modular network (4 modules) ---")
# res_mod = computational_capacity(A_mod, n_trials=2, T_train=1000, T_test=300)
# print(f"  Memory capacity:     {res_mod['memory_capacity_total']:.2f}")
# print(f"  Nonlinear capacity:  {res_mod['nonlinear_capacity_total']:.2f}")
# print(f"  Memory/NL ratio:     {res_mod['memory_nonlinear_ratio']:.3f}")
# print(f"  State dimensionality:{res_mod['state_dimensionality']:.1f}")
# print(f"  Lyapunov exponent:   {res_mod['lyapunov_exponent']:.4f}")

# print("\n" + "=" * 60)
# print("Capacity vectors (for embedding):")
# print(f"  Random:  {res_rand['capacity_vector']}")
# print(f"  Ring:    {res_ring['capacity_vector']}")
# print(f"  Modular: {res_mod['capacity_vector']}")
# print("=" * 60)


Validating computational_capacity on test networks

--- Random ER network (p=0.1) ---


/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_42870/2019635556.py:121: RuntimeWarning: divide by zero encountered in matmul
  x_new = np.tanh(W @ x + u + bias)
/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_42870/2019635556.py:121: RuntimeWarning: overflow encountered in matmul
  x_new = np.tanh(W @ x + u + bias)
/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_42870/2019635556.py:121: RuntimeWarning: invalid value encountered in matmul
  x_new = np.tanh(W @ x + u + bias)
/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_42870/2019635556.py:135: RuntimeWarning: divide by zero encountered in matmul
  W = linalg.solve(X.T @ X + alpha * np.eye(n), X.T @ Y, assume_a='pos')
/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_42870/2019635556.py:135: RuntimeWarning: overflow encountered in matmul
  W = linalg.solve(X.T @ X + alpha * np.eye(n), X.T @ Y, assume_a='pos')
/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_42870/201963555

{'memory_capacity_total': 8.517169992495042,
 'memory_timescale': 10.0,
 'nonlinear_capacity_total': 3.416133677752688,
 'cubic_capacity_total': 7.227938454330771,
 'cross_capacity_total': 7.663980972877067,
 'memory_nonlinear_ratio': 0.7137311031252949,
 'total_capacity': 26.825223097455567,
 'state_dimensionality': 1.377022116968321,
 'state_entropy': 0.5524952114120831,
 'state_rank': 100,
 'separation_ratio': 2.0529676933979157,
 'lyapunov_exponent': -0.09852869512938156}